In [1]:
%pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 4.3 MB/s eta 0:00:00


In [2]:
import os
os.environ['GROQ_API_KEY'] = "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"

from groq import Groq
client = Groq(api_key=os.environ['GROQ_API_KEY'])
completion = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {
            "role": "user",
            "content": "Explique a importância da IA generativa para o Sistema Judiciário Brasileiro."
        }
    ]
)
print(completion.choices[0].message.content)

A inteligência artificial (IA) generativa pode ter um impacto significativo no Sistema Judiciário Brasileiro, trazendo diversas vantagens e melhorias para o funcionamento do sistema. Aqui estão algumas da importância da IA generativa para o Sistema Judiciário Brasileiro:

1. **Automatização de Tarefas**: A IA generativa pode ser utilizada para automatizar tarefas rotineiras e repetitivas, como a elaboração de documentos, a análise de dados e a identificação de padrões. Isso pode liberar tempo e recursos para que os magistrados e servidores possam se concentrar em tarefas mais complexas e estratégicas.

2. **Análise de Dados**: A IA generativa pode ser utilizada para analisar grandes volumes de dados, identificar padrões e tendências, e fornecer insights valiosos para a tomada de decisões. Isso pode ser particularmente útil em casos que envolvem grandes volumes de dados, como processos de improbidade administrativa ou crimes financeiros.

3. **Elaboração de Decisões**: A IA generativa p

In [3]:
class Agent:
  def __init__(self, client, system):
    self.client = client
    self.system = system
    self.messages = []
    if self.system is not None:
      self.messages.append({"role": "system", "content": self.system})

  def __call__(self, message=""):
    if message:
      self.messages.append({"role": "user", "content": message})
    result = self.execute()
    self.messages.append({"role": "assistant", "content": result})
    return result

  def execute(self):
    completion = client.chat.completions.create(
      model="llama-3.3-70b-versatile",
      messages=self.messages
    )
    return completion.choices[0].message.content

In [4]:
system_prompt="""
Você é um agente ReAct, e deve executar um loop de Thought, Action, PAUSE, Observation.
Ao final do loop, você imprime uma resposta (Answer)
Use Thought para descrever os seus pensamentos sobre a questão que foi perguntada a você
Use Action para executar uma das ações disponíveis para você - e então retorne PAUSE.
Observation será o resultado da execução das ações (Action)

Suas ações disponíveis são:

calculate:
exemplo: calcule 25 * 8 / 9
Execute um cálculo e retorne o número. Use Python para garantir o uso da sintaxe de ponto flutuante, caso necessário.

get_number_pets_breed:
exemplo: get_number_pets_breed: gatos
Retorne a quantidade máxima de raças de gatos já registrada
exemplo 2: get_number_pets_breed: cachorros
Retorne a quantidade máxima de raças de cachorros já registrada

exemplo da sessão:

Pergunta: Qual é a soma da quantidade máxima de raças de gatos e de cachorros já registradas?

Thought: Eu preciso encontrar a quantidade máxima de raças de gatos já registrada.
Action: get_number_pets_breed:gatos
PAUSE

Você deverá retornar com isso
Observation: A quantidade máxima de raças de gatos já registrada é 73

Você deverá retornar com isso
Thought: Eu preciso encontrar a quantidade máxima de raças de cachorros já registrada.
Action: get_number_pets_breed:cachorros
PAUSE

Você deverá retornar com isso
Observation: A quantidade máxima de raças de cachorros já registrada é 344

Se você já tem a resposta, imprima a resposta (Answer)
Answer: A soma da quantidade máxima de raças de gatos e de cachorros já registrada é 415.

""".strip()

In [5]:
#Ferramentas

def calculate(operation: str) -> float:
  return eval(operation)

def get_number_pets_breed(pet) -> float:
  match pet.lower():
    case "gatos":
      return 73
    case "cachorros":
      return 344
    case _:
      return 0

In [6]:
Buddy = Agent(client=client, system=system_prompt)

In [17]:
import re

def agent_loop(max_iterations, system, query):
  agent = Agent(client, system_prompt)
  tools = ['calculate', 'get_number_pets_breed']
  next_prompt = query

  i=0
  while i < max_iterations:
    i += 1
    result = agent(next_prompt)
    print(result)

    if "PAUSE" in result and "Action" in result:
      action_match = re.search(r"Action: ([a-z_]+)(?::)?\s*(.+?)\s*PAUSE", result, re.IGNORECASE)
      #tive que pedir ajuda para a IA para adaptar esse regex
      #e tbm a chamada das funções (ferramentas) de modo correto

      if action_match:
        chosen_tool = action_match.group(1)
        arg = action_match.group(2)

        if chosen_tool in tools:
          tool_function = globals()[chosen_tool]
          result_tool = tool_function(arg)
          next_prompt = f"Observation: {result_tool}"

        else:
          next_prompt = "Observation: Ferramenta não encontrada"
      else:
        next_prompt = "Observation: Formato de ação inesperado"

      print(next_prompt)
      continue

    if "Answer" in result:
      break

In [16]:
agent_loop(max_iterations=10, system=system_prompt, query="Qual é a soma da quantidade máxima de raças de gatos e de cachorros já registradas?")

Thought: Eu preciso encontrar a quantidade máxima de raças de gatos já registrada e a quantidade máxima de raças de cachorros já registrada para somá-las.
Action: get_number_pets_breed:gatos
PAUSE
Observation: 73
Thought: Agora que eu tenho a quantidade máxima de raças de gatos, eu preciso encontrar a quantidade máxima de raças de cachorros já registrada.
Action: get_number_pets_breed:cachorros
PAUSE
Observation: 344
Thought: Agora que eu tenho as quantidades máximas de raças de gatos e cachorros, eu posso somá-las para encontrar a resposta.
Action: calculate 73 + 344
PAUSE
Observation: 417
Thought: Com o resultado da soma, posso agora fornecer a resposta final.
Answer: A soma da quantidade máxima de raças de gatos e de cachorros já registrada é 417.
